# ML-3: Predictive Maintenance Model Training & Evaluation

This notebook trains and evaluates binary classification models to predict **Machine failure** using the cleaned AI4I 2020 Predictive Maintenance Dataset.

## Objectives:
1. Ingest cleaned telemetry dataset.
2. Construct physical domain features (Temperature Difference $\Delta T$, Mechanical Power $P_{\text{mech}}$, Overstrain Index).
3. Apply a fixed 80/20 Stratified Train/Test split (`random_state=42`).
4. Fit preprocessing (`ColumnTransformer`: OneHotEncoder + StandardScaler) **strictly on training data**.
5. Train 4 Candidate Models: **Logistic Regression**, **Decision Tree**, **Random Forest**, and **XGBoost**.
6. Evaluate models via **5-Fold Stratified K-Fold Cross-Validation** on the training set.
7. Evaluate final candidate models on the unseen holdout test set (ROC-AUC, PR-AUC, Precision, Recall, F1).
8. Save the best candidate model artifact for **ML-4 (Explainable AI / SHAP Attribution)**.

In [2]:
import os
import sys
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_score, 
    recall_score, f1_score, confusion_matrix, classification_report
)

# Append src module to path
sys.path.append(os.path.abspath("../src"))
from feature_engineering import DomainFeatureEngineer, build_preprocessing_pipeline

## 1. Load Data & Apply Feature Engineering

In [3]:
# Load cleaned dataset
data_path = "../../data/cleaned_predictive_maintenance.csv"
if not os.path.exists(data_path):
    data_path = "../../data/ai4i2020.csv"

df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape}")

# Exclude Identifiers & Failure Mode Targets to prevent data leakage
target_col = 'Machine failure'
excluded_cols = ['UDI', 'Product ID', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']
feature_cols = [c for c in df.columns if c not in excluded_cols and c != target_col]

X = df[feature_cols].copy()
y = df[target_col].copy()

# Domain Feature Construction
engineer = DomainFeatureEngineer()
X_eng = engineer.transform(X)
print(f"Engineered Features Count: {X_eng.shape[1]}")

Dataset Shape: (100, 7)
Engineered Features Count: 9


## 2. Fixed 80/20 Stratified Split & Preprocessing

In [4]:
categorical_cols = ['Type']
numerical_cols = [c for c in X_eng.columns if c not in categorical_cols]

# Stratified 80/20 Train/Test Split
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_eng, y, test_size=0.2, random_state=42, stratify=y
)

# Fit preprocessor strictly on training data
preprocessor = build_preprocessing_pipeline(categorical_cols, numerical_cols)
X_train_proc = preprocessor.fit_transform(X_train_raw)
X_test_proc = preprocessor.transform(X_test_raw)

feature_names = preprocessor.get_feature_names_out()
X_train = pd.DataFrame(X_train_proc, columns=feature_names)
X_test = pd.DataFrame(X_test_proc, columns=feature_names)

print(f"X_train Shape: {X_train.shape}")
print(f"X_test Shape:  {X_test.shape}")

X_train Shape: (80, 11)
X_test Shape:  (20, 11)


## 3. Candidate Models & 5-Fold Stratified Cross-Validation

In [6]:
import re

# Clean feature names for XGBoost
X_train = X_train.copy()
X_train.columns = [
    re.sub(r"[\[\]<>]", "_", str(col))
    for col in X_train.columns
]

X_test = X_test.copy()
X_test.columns = [
    re.sub(r"[\[\]<>]", "_", str(col))
    for col in X_test.columns
]

# Check class distribution
print("Training class distribution:")
print(y_train.value_counts())

neg_count = np.sum(y_train == 0)
pos_count = np.sum(y_train == 1)

scale_pos_weight = neg_count / pos_count

models = {
    "Logistic Regression": LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced",
        max_depth=6,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        max_depth=10,
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        n_estimators=100,
        max_depth=5,
        scale_pos_weight=scale_pos_weight,
        learning_rate=0.05,
        random_state=42,
        eval_metric="logloss"
    )
}

skf = StratifiedKFold(
    n_splits=2,
    shuffle=True,
    random_state=42
)

scoring = [
    "roc_auc",
    "average_precision",
    "f1",
    "precision",
    "recall"
]

cv_summary = []
trained_models = {}

for name, model in models.items():

    cv_res = cross_validate(
        model,
        X_train,
        y_train,
        cv=skf,
        scoring=scoring
    )

    cv_summary.append({
        "Model": name,
        "CV ROC-AUC": (
            f"{np.mean(cv_res['test_roc_auc']):.4f} "
            f"± {np.std(cv_res['test_roc_auc']):.4f}"
        ),
        "CV PR-AUC": (
            f"{np.mean(cv_res['test_average_precision']):.4f}"
        ),
        "CV F1-Score": (
            f"{np.mean(cv_res['test_f1']):.4f}"
        ),
        "CV Precision": (
            f"{np.mean(cv_res['test_precision']):.4f}"
        ),
        "CV Recall": (
            f"{np.mean(cv_res['test_recall']):.4f}"
        )
    })

    model.fit(X_train, y_train)
    trained_models[name] = model

pd.DataFrame(cv_summary)

Training class distribution:
Machine failure
0    78
1     2
Name: count, dtype: int64


C:\Users\bingu\AppData\Roaming\Python\Python314\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\bingu\AppData\Roaming\Python\Python314\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\bingu\AppData\Roaming\Python\Python314\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape

,Model,CV ROC-AUC,CV PR-AUC,CV F1-Score,CV Precision,CV Recall
0,Logistic Regression,0.0000 ± 0.0000,0.0250,0.0000,0.0000,0.0000
1,Decision Tree,0.5000 ± 0.0000,0.0250,0.0000,0.0000,0.0000
2,Random Forest,0.8077 ± 0.0385,0.1131,0.0000,0.0000,0.0000
3,XGBoost,0.5000 ± 0.0000,0.0250,0.0000,0.0000,0.0000


## 4. Unseen Holdout Test Set Evaluation

In [7]:
test_results = []
best_model_name = None
best_auc = -1.0

for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    auc = roc_auc_score(y_test, y_prob)
    pr_auc = average_precision_score(y_test, y_prob)
    f1 = f1_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    
    test_results.append({
        "Model": name,
        "Test ROC-AUC": round(auc, 4),
        "Test PR-AUC": round(pr_auc, 4),
        "Test F1-Score": round(f1, 4),
        "Test Precision": round(prec, 4),
        "Test Recall": round(rec, 4)
    })
    
    if auc > best_auc:
        best_auc = auc
        best_model_name = name

test_df = pd.DataFrame(test_results)
print(f"Selected Candidate Model: {best_model_name} (Test ROC-AUC: {best_auc:.4f})")
test_df

Selected Candidate Model: Logistic Regression (Test ROC-AUC: 0.8947)


C:\Users\bingu\AppData\Roaming\Python\Python314\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\bingu\AppData\Roaming\Python\Python314\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\bingu\AppData\Roaming\Python\Python314\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape

,Model,Test ROC-AUC,Test PR-AUC,Test F1-Score,Test Precision,Test Recall
0,Logistic Regression,0.8947,0.3333,0.0,0.0,0.0
1,Decision Tree,0.4737,0.0500,0.0,0.0,0.0
2,Random Forest,0.7895,0.2000,0.0,0.0,0.0
3,XGBoost,0.7632,0.1429,0.0,0.0,0.0


## 5. Export Best Model Artifact for ML-4 Explainability

In [8]:
models_dir = "../models"
os.makedirs(models_dir, exist_ok=True)

best_model_path = os.path.join(models_dir, "best_model.joblib")
joblib.dump(trained_models[best_model_name], best_model_path)
print(f"Best model ({best_model_name}) saved to: {best_model_path}")


Best model (Logistic Regression) saved to: ../models\best_model.joblib
